In [1]:
!pip install pydicom

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------------------------- ------------- 1.6/2.4 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 9.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Aryan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import pydicom
import numpy as np

In [3]:
def process_patient_ct(patient_folder_path):
    """
    Reads a patient's directory of DICOM files, sorts them into a 3D volume,
    converts to Hounsfield Units (HU), and applies a Lung Window.
    """
    # ---------------------------------------------------------
    # PART A: EXTRACT & SORT
    # ---------------------------------------------------------
    
    # 1. Get all .dcm files in the specific patient folder (e.g., 'C001/')
    dicom_files = [os.path.join(patient_folder_path, f) 
                   for f in os.listdir(patient_folder_path) if f.endswith('.dcm')]
    
    # 2. Read all files into a list
    slices = [pydicom.dcmread(f) for f in dicom_files]
    
    # 3. CRITICAL: Sort slices mathematically by their Z-axis position 
    # (If you don't do this, your 2.5D stacking will mix up random parts of the lungs!)
    slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    
    # Stack the raw pixel arrays into a single 3D numpy volume
    image_volume = np.stack([s.pixel_array for s in slices])
    
    # ---------------------------------------------------------
    # PART B: HOUNSFIELD UNIT (HU) CONVERSION
    # ---------------------------------------------------------
    # DICOM pixels are stored as raw scanner outputs. We must convert them to standard HU.
    # We grab the Rescale Intercept and Slope from the first slice's metadata.
    intercept = slices[0].RescaleIntercept
    slope = slices[0].RescaleSlope
    
    # Apply the linear conversion formula
    if slope != 1:
        image_volume = slope * image_volume.astype(np.float64)
        image_volume = image_volume.astype(np.int16)
        
    hu_volume = image_volume + np.int16(intercept)
    
    # ---------------------------------------------------------
    # PART C: LUNG WINDOWING
    # ---------------------------------------------------------
    # We apply the specific Lung Window (e.g., -1250 to +250) to isolate lung tissue
    MIN_BOUND = -1250.0
    MAX_BOUND = 250.0
    
    # Clip all values outside this range
    windowed_volume = np.clip(hu_volume, MIN_BOUND, MAX_BOUND)
    
    # Optional: Normalize the windowed volume to a 0.0 to 1.0 range for the neural network
    normalized_volume = (windowed_volume - MIN_BOUND) / (MAX_BOUND - MIN_BOUND)
    
    return normalized_volume

# --- Example Usage ---
# path = f"{dataset_path}/Dataset-S1/COVID-S1/C001"
# patient_3d_volume = process_patient_ct(path)
# print(patient_3d_volume.shape) # Output will be something like (Num_Slices, 512, 512)

---

### **The Setup & Imports**

* **`import os`**: Helps your code navigate your computer's folders and files.
* **`import pydicom`**: The standard library for reading `.dcm` (DICOM) medical files. It reads both the pixel data and the hidden patient/scanner metadata.
* **`import numpy as np`**: Used for heavy mathematical operations and creating 3D matrices (arrays).

---

### **PART A: Extract & Sort (Data nikalna aur arrange karna)**

* **`dicom_files = [os.path.join(...) for f in os.listdir(...) if f.endswith('.dcm')]`**
* This goes into the `C001` patient folder, looks at every file, and if it ends with `.dcm`, it saves the full file path into a list.


* **`print("Dicom Files :",type(dicom_files[0]))`**
* Just a check. It will print `<class 'str'>` because right now, it's just a list of text paths.


* **`slices = [pydicom.dcmread(f) for f in dicom_files]`**
* This is where the magic starts. `pydicom.dcmread` actually opens each file and loads all the CT scan data and metadata into memory.


* **`slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))`**
* **CRITICAL STEP:** A CT scan is taken as a series of 2D slices. When you read files from a folder, the computer might load slice #40 before slice #2. If you stack them randomly, the lungs will look like a jigsaw puzzle. `ImagePositionPatient[2]` grabs the **Z-axis** (physical height inside the scanner) from the metadata and sorts the slices from top to bottom (or bottom to top).


* **`image_volume = np.stack([s.pixel_array for s in slices])`**
* This extracts just the pictures (`pixel_array`) from the DICOM files and stacks them like a deck of cards into a single 3D NumPy array.



---

### **PART B: Hounsfield Unit (HU) Conversion**

Raw CT scanners don't output standard colors; they output arbitrary electrical signals. We have to convert these raw numbers into a universal medical standard called **Hounsfield Units (HU)**, where water is exactly 0 and air is -1000.

* **`intercept = slices[0].RescaleIntercept`** and **`slope = slices[0].RescaleSlope`**
* Every scanner embeds a secret "calibration key" in the DICOM metadata. These are the intercept and slope.


* **`if slope != 1: ...`** and **`hu_volume = image_volume + np.int16(intercept)`**
* This applies the linear conversion formula:
$HU = (\text{Pixel Value} \times \text{Slope}) + \text{Intercept}$
* Now, every single pixel in your 3D volume represents true tissue density, regardless of what brand of CT scanner was used.



---

### **PART C: Lung Windowing (Sirf Lungs par focus karna)**

A CT scan captures everything: bones, fat, blood, and air. "Windowing" is like putting on a specific pair of sunglasses that blocks out everything except the organ you want to see.

* **`MIN_BOUND = -1250.0`** and **`MAX_BOUND = 250.0`**
* In the HU scale, lung tissue and air generally fall between -1250 and +250.


* **`windowed_volume = np.clip(hu_volume, MIN_BOUND, MAX_BOUND)`**
* Any tissue denser than +250 (like bones) is forced to become exactly +250 (pure white).
* Any tissue lighter than -1250 (outside air) is forced to become exactly -1250 (pure black).
* This leaves only the intricate details of the lung tissue visible.


* **`normalized_volume = (windowed_volume - MIN_BOUND) / (MAX_BOUND - MIN_BOUND)`**
* Neural Networks (like CNNs) hate weird numbers like -1250. They love numbers between `0.0` and `1.0`.
* This is Min-Max scaling. It squashes your entire lung window down into a simple 0 to 1 scale so a deep learning model can process it efficiently.


* **`return normalized_volume`**
* The function hands back your perfectly cleaned, stacked, and normalized 3D representation of the patient's lungs.



---

In [5]:
# --- 2. The New Batch Processing Function ---
def batch_process_and_save(source_directory, output_directory):
    """
    Loops through all patient folders in a directory, processes them, 
    and saves them as fast-loading .npy files.
    """
    # Create the output folder if it doesn't exist yet
    os.makedirs(output_directory, exist_ok=True)
    
    # Get a list of all items in the source folder (C001, C002, C003...)
    patient_folders = os.listdir(source_directory)
    
    for patient_id in patient_folders:
        patient_path = os.path.join(source_directory, patient_id)
        
        # Check to make sure it's actually a folder and not a hidden file
        if os.path.isdir(patient_path):
            print(f"Processing patient: {patient_id}...")
            
            try:
                # 1. Process the raw DICOMs into a clean 3D volume
                processed_volume = process_patient_ct(patient_path)
                
                # 2. Define exactly where to save it and what to call it
                # Example: "Processed_Data/COVID-S1/C001.npy"
                save_path = os.path.join(output_directory, f"{patient_id}.npy")
                
                # 3. Save the array to your hard drive
                np.save(save_path, processed_volume)
                print(f" --> Saved successfully! Shape: {processed_volume.shape}")
                
            except Exception as e:
                print(f" --> FAILED to process {patient_id}. Error: {e}")

# --- 3. Run the Code based on your folder structure ---
# Point this to where your raw DICOM folders live
raw_data_source = "Dataset-S1/COVID-S1"

# Point this to a brand new folder where you want to store the clean data
clean_data_output = "CovidDataset_Processed/Processed_Data/COVID-S1"

# Execute the loop!
batch_process_and_save(raw_data_source, clean_data_output)

Processing patient: C001...
 --> Saved successfully! Shape: (143, 512, 512)
Processing patient: C002...
 --> Saved successfully! Shape: (144, 512, 512)
Processing patient: C003...
 --> Saved successfully! Shape: (163, 512, 512)
Processing patient: C004...
 --> Saved successfully! Shape: (125, 512, 512)
Processing patient: C005...
 --> Saved successfully! Shape: (122, 512, 512)
Processing patient: C006...
 --> Saved successfully! Shape: (139, 512, 512)
Processing patient: C007...
 --> Saved successfully! Shape: (144, 512, 512)
Processing patient: C008...
 --> Saved successfully! Shape: (122, 512, 512)
Processing patient: C009...
 --> Saved successfully! Shape: (141, 512, 512)
Processing patient: C010...
 --> Saved successfully! Shape: (134, 512, 512)
Processing patient: C011...
 --> Saved successfully! Shape: (139, 512, 512)
Processing patient: C012...
 --> Saved successfully! Shape: (144, 512, 512)
Processing patient: C013...
 --> Saved successfully! Shape: (144, 512, 512)
Processing p